## Tabela: `workspace.silver.meta_alfabetizacao_municipio`

| Item | Detalhe |
|------|---------|
| **Tabela** | `workspace.silver.meta_alfabetizacao_municipio` |
| **Origem** | `workspace.bronze.meta_alfabetizacao_municipio_raw` |
| **Descrição** | Tratamento e padronização dos dados de metas de alfabetização por município, previamente ingeridos na camada Bronze. |
| **Camada** | Silver (tratamento e padronização) |
| **Colunas de controle** | `data_ingestao` (timestamp de carga), `fonte` (origem dos dados) |
| **Criado por** | suelen.ramos |
| **Data de criação** | 2025-01-20 |

### Chave de negócio
A combinação `ano + id_municipio + rede_codigo` forma a chave única da tabela.

### Tratamentos realizados

- Padronização dos tipos de dados;
- Remoção de espaços;
- Padronização dos nomes das colunas;
- Preservação das informações originais;
- Preservação das colunas de controle da ingestão;
- Validação de duplicidades após a transformação.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.silver.meta_alfabetizacao_municipio
USING DELTA
AS
SELECT
    CAST(m.ano AS INT) AS ano,
    TRIM(CAST(m.id_municipio AS STRING)) AS id_municipio,

    TRIM(CAST(m.rede AS STRING)) AS rede,
    CASE
        WHEN LOWER(TRIM(CAST(m.rede AS STRING))) = 'municipal' THEN '3'
        ELSE NULL
    END AS rede_codigo,

    CAST(m.taxa_alfabetizacao AS DOUBLE) AS taxa_alfabetizacao,

    CAST(m.meta_alfabetizacao_2024 AS DOUBLE) AS meta_alfabetizacao_2024,
    CAST(m.meta_alfabetizacao_2025 AS DOUBLE) AS meta_alfabetizacao_2025,
    CAST(m.meta_alfabetizacao_2026 AS DOUBLE) AS meta_alfabetizacao_2026,
    CAST(m.meta_alfabetizacao_2027 AS DOUBLE) AS meta_alfabetizacao_2027,
    CAST(m.meta_alfabetizacao_2028 AS DOUBLE) AS meta_alfabetizacao_2028,
    CAST(m.meta_alfabetizacao_2029 AS DOUBLE) AS meta_alfabetizacao_2029,
    CAST(m.meta_alfabetizacao_2030 AS DOUBLE) AS meta_alfabetizacao_2030,

    CAST(m.nivel_alfabetizacao AS INT) AS nivel_alfabetizacao,
    CAST(m.percentual_participacao AS DOUBLE) AS percentual_participacao,

    m.data_ingestao,
    m.fonte,
    m.lote_id

FROM workspace.bronze.meta_alfabetizacao_municipio_raw m

WHERE m.ano IS NOT NULL
  AND m.id_municipio IS NOT NULL

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY m.ano, m.id_municipio
    ORDER BY m.data_ingestao DESC
) = 1
""")

print("Tratamento das metas municipais atualizado!")

In [0]:
registrar_execucao("SILVER", "meta_municipio", _m, row_count=contar_tabela("workspace.silver.meta_alfabetizacao_municipio"))